<h1><center>Digital Twin Project</center></h1>

In [1]:
# Import Libraries
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
# The usual start
load_dotenv() # Load environment variables from .env file
openai = OpenAI() # Initialize OpenAI client

In [4]:
# For Pushover
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and valid.")
    else:
        print("Pushover user found but doesn't start with a u")
else:
    print("Pushover user not found in environment variables.")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and valid.")
    else:
        print("Pushover token found but doesn't start with an a")
else:
    print("Pushover token not found in environment variables.")
    

Pushover user found and valid.
Pushover token found and valid.


In [5]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, 
               "token": pushover_token,
               "message": message}
    requests.post(pushover_url, data = payload)
    

In [6]:
push("Hey!!!")

Push: Hey!!!


In [7]:
def record_user_details(email, name = "Name not provided", notes = "not provided"):
    push(f"Recording interest from {name} with email {email}, and notes {notes}")
    return "OK"

In [14]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "Ok"

In [15]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being contacted and provided an email address.",
    "parameters":{
        "type": "object",
        "properties":{
            "email": {"type": "string",
                      "description": "The email address of the user."},
            "name": {"type": "string",
                     "description": "The name of the user."},
            "notes": {"type": "string",
                      "description": "Any notes the user provided about the conversation or their interest."}

        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [16]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any questions that cannot be answered by the agent.",
    "parameters":{
        "type": "object",
        "properties":{
            "question": {"type": "string", 
                      "description": "The question that cannot be answered."},
        },
        "required": ["question"],
        "additionalProperties": False
    }

}

In [17]:
tools = [{"type": "function",
          "function": record_user_details_json},
          {"type": "function",
           "function": record_unknown_question_json}]

In [18]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being contacted and provided an email address.',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of the user.'},
     'name': {'type': 'string', 'description': 'The name of the user.'},
     'notes': {'type': 'string',
      'description': 'Any notes the user provided about the conversation or their interest.'}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': 'Always use this tool to record any questions that cannot be answered by the agent.',
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': 'The question that cannot be answered.'}},
    'required': ['question'],
    'additionalProperties': False}}}]

In [19]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush = True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)
    
        results.append({"role": "tool",
                        "content": json.dumps(result), 
                        "tool_call_id": tool_call.id})
    return results

## Using Python built-in globals()


In [20]:
globals()["record_unknown_question"]("This is a really hard question")


Push: Recording This is a really hard question asked that I couldn't answer


'Ok'

In [21]:
# This gives us a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for  tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush = True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found" 
        results.append({"role": "tool",
                        "content": json.dumps(result),
                        "tool_call_id": tool_call.id})
    return results

In [ ]:
reader = PdfReader("my twin/LinkedIn.pdf")
linkedin ="" 
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text
with open("my twin/summary.txt", 
          "r",
          encoding = "utf-8")as f:
    summary = f.read()

In [23]:
system_prompt = f"""

# Your role

You are a digital twin running a website that is designed 
to answer questions about an individual. You represent the
You answer questions related to their career, experience, 
background, education, projects, and relevant professional 
information.

Here are the details of the person you are representing:
{summary}

If asked, you explain clearly that you are an AI digital 
twin and not the actual person. 

# Context
Here is a summary of the person's LinkedIn profile so that
you can answer questions. 

{linkedin}

# Rules

Engage with users in a professional and courteous manner. 
Focus solely on the professional aspects of the individual.
If questions arise outside the scope of the individual's
professional background, politely inform them that you 
can only provide information related to the individual's
professional background.

# IMPORTANT: If you are asked a question that you do not 
know or have the answer to use the tool to record the 
question. Do not attempt to answer questions that are outside
the scope of the individual's professional background.
Always use the tool to record questions that you cannot answer.
 

 """

In [24]:
def chat(message, history):
    messages = [{
        "role": "system",
        "content": system_prompt}] + \
        history + \
        [{"role": "user",
          "content": message}]
    response = openai.chat.completions.create(
        model = "gpt-5.4-mini",
        messages = messages,
        tools = tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model = "gpt-5.4-mini",
                                                  messages = messages,
                                                  tools = tools)
    return response.choices[0].message.content

In [25]:
gr.ChatInterface(fn = chat).launch(inbrowser = True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/Users/michaelperrine/Documents/ai-engineer-agentic-track-donner/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/michaelperrine/Documents/ai-engineer-agentic-track-donner/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/michaelperrine/Documents/ai-engineer-agentic-track-donner/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/michaelperrine/Documents/ai-engineer-agentic-track-donner/agent

Tool called: record_unknown_question
Push: Recording What is your favorite food? asked that I couldn't answer


/Users/michaelperrine/Documents/ai-engineer-agentic-track-donner/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/michaelperrine/Documents/ai-engineer-agentic-track-donner/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/michaelperrine/Documents/ai-engineer-agentic-track-donner/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/michaelperrine/Documents/ai-engineer-agentic-track-donner/agent